In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

def prepare():
    module_path = os.path.abspath(os.path.join('..'))
    if module_path not in sys.path:
        sys.path.append(module_path)

In [3]:
import torch
import numpy as np
prepare()
from exp_labelcert_binaryclass import run

In [4]:
model_params = dict(
    label = "GraphSAGE", 
    model = "GCN", 
    normalization = "graph_sage_normalization",
    activation = "relu",
    depth = 1,
    regularizer = 0.001,
    pred_method = "svm",
    bias = False,
    alpha_tol = 1e-4,
    solver = "qplayer",
)

certificate_params = dict(
    delta = 0.0,
    TimeLimit = 86400,
    LogToConsole = 1,
    OutputFlag = 1,
    Threads = 2,
    Presolve = 2
)

verbosity_params = dict(
    debug_lvl = "warning"
)  

other_params = dict(
    device = "0",
    dtype = torch.float64,
    allow_tf32 = False,
    path_gurobi_license = "path/to/your/gurobi/license"
)

In [5]:
data_params = dict(
    dataset = "csbm",
    learning_setting = "transductive", 
    specification = dict(
        classes = 2,
        n_trn_labeled = 10,
        n_trn_unlabeled = 0,
        n_val = 10,
        n_test = 180,
        sigma = 1,
        avg_within_class_degree = 1.58 * 2,
        avg_between_class_degree = 0.37 * 2,
        K = 1.5,
        seed = 0 # used to generate the dataset & data split
    )
)

In [6]:
import pandas as pd
import time

seeds = [0, 1, 2, 3, 4]
delta = 0.0
certificate_params["delta"] = delta

metrics = [
    "accuracy_test",
    "accuracy_trn",
    "accuracy_cert_pois_robust",
    "accuracy_cert_pois_unrobust",
    "delta",
]

summary = []

for seed in seeds:
    data_params["specification"]["seed"] = seed
    
    start_time = time.time()
    result = run(data_params, model_params, certificate_params, verbosity_params, other_params, seed)
    end_time = time.time()
    runtime = round(end_time - start_time, 2)
    
    summary.append({k: result[k] for k in metrics} | {"runtime": runtime})
    
    
df = pd.DataFrame(summary, index=[f"Seed {s}" for s in seeds])
df.index.name = "seed"
df.to_csv(f'results/samplewise/csbm-{delta:.2f}.csv', index=True)
df

CSBM mu:
[0.28347334 0.28347334 0.28347334 0.28347334 0.28347334 0.28347334
 0.28347334]
20 alphas found: ['0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010', '0.0010']
Set parameter Username
Set parameter LicenseID to value 2663332
Academic license - for non-commercial use only - expires 2026-05-10
Set parameter BestObjStop to value 0
Set parameter BestBdStop to value 0
Set parameter IntegralityFocus to value 1
Set parameter IntFeasTol to value 0.0001
Set parameter DualReductions to value 0
Set parameter Presolve to value 2
Set parameter Threads to value 2
Set parameter FeasibilityTol to value 0.0001
Set parameter OptimalityTol to value 0.0001
Set parameter TimeLimit to value 86400
Gurobi Optimizer version 11.0.1 build v11.0.1rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: AMD Ryzen 7 4800H with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thr

,accuracy_test,accuracy_trn,accuracy_cert_pois_robust,accuracy_cert_pois_unrobust,delta,runtime
seed,,,,,,
Seed 0,0.905556,1.00,1.0,0,0.0,19.70
Seed 1,0.811111,0.95,1.0,0,0.0,19.52
Seed 2,0.794444,0.90,1.0,0,0.0,22.04
Seed 3,0.827778,1.00,1.0,0,0.0,23.20
Seed 4,0.838889,0.90,1.0,0,0.0,30.94
